# NutraMetrics: Разведочный анализ данных (EDA) и проверка статистических гипотез

**Автор:** R&D Formulation Chemist (6+ лет в FoodTech/Nutraceuticals *Nuversys* и Фармацевтике *Биннофарм Групп*) & Product Data Analyst

В данном ноутбуке проводится статистическое исследование **907 пероральных препаратов (БАДов и витаминов)**, собранных с живого аптечного рынка РФ (`Uteka.ru`, `Apteka.ru`, `Zdravcity.ru`, `Eapteka.ru`), и проверяются **5 продуктовых и химико-технологических гипотез** с использованием непараметрических критериев (`Kruskal-Wallis`, `Mann-Whitney U`, `Friedman`, `Wilcoxon`), поправки Холма–Бонферрони и двухфакторного дисперсионного анализа (`Type-II ANOVA`).

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

df = pd.read_csv('../data/processed/nutrametrics_priced.csv')
print(f'Размер очищенного датасета: {df.shape[0]} SKU x {df.shape[1]} признаков')
df[['nutra_category', 'price_rub', 'cost_per_day_rub', 'bap_dose_rub', 'nvi_score', 'brand_overpay_pct']].describe().round(2)

Размер очищенного датасета: 907 SKU x 79 признаков

       price_rub  cost_per_day_rub  bap_dose_rub  nvi_score  brand_overpay_pct
count     907.00            907.00        907.00     907.00             907.00
mean      827.74             24.60         19.73      57.77              16.88
std       813.14             31.42         28.81      19.86              73.02
min        50.00              0.21          0.16      14.00             -94.80
25%       345.50              7.46          6.12      43.80             -30.60
50%       571.00             15.37         11.30      58.20               0.00
75%      1018.50             29.38         22.41      72.10              48.05
max      8468.00            381.40        480.07      98.00             686.40

## H1. Парадокс биодоступности: R&D Tier 3 vs Tier 1/2 (C_day vs BAP-Dose & NVI)

- **Нулевая гипотеза ($H_0$):** H0: Номинальная цена дня курса (C_day), эффективная биодоступная цена (BAP-Dose) и индекс NVI одинаково распределены между Tier 1, Tier 2 и Tier 3.
- **Статистический критерий:** `Kruskal-Wallis H-test + попарный Mann-Whitney U (поправка Холма–Бонферрони)`
- **Вердикт:** **H0 отвергается (p < 0.0001)**
- **R&D и Бизнес-вывод:** Номинально день приема Tier 3 дороже Tier 1 (17.32 ₽ vs 11.27 ₽, H=40.2, p=1.88e-09). Однако за счет высокой биодоступности (K_bio=1.75) и адекватных дозировок эффективная стоимость BAP-Dose у Tier 3 (9.14 ₽, 95% ДИ [8.31; 10.89]) статистически значимо НИЖЕ, чем у Tier 2 (12.96 ₽, p_holm=3.56e-06) и выигрывает по среднему индексу ценности NVI (73.1 против 43.9 у Tier 1, H=268.7, p=4.41e-59).

In [2]:
# Результаты статистического теста H1
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H1']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H1",
  "title": "Парадокс биодоступности: R&D Tier 3 vs Tier 1/2 (C_day vs BAP-Dose & NVI)",
  "null_hypothesis": "H0: Номинальная цена дня курса (C_day), эффективная биодоступная цена (BAP-Dose) и индекс NVI одинаково распределены между Tier 1, Tier 2 и Tier 3.",
  "test_used": "Kruskal-Wallis H-test + попарный Mann-Whitney U (поправка Холма–Бонферрони)",
  "kruskal_c_day_H": 40.19,
  "kruskal_c_day_p": 1.875737266452031e-09,
  "kruskal_bap_H": 24.73,
  "kruskal_bap_p": 4.272862647465461e-06,
  "kruskal_nvi_H": 268.74,
  "kruskal_nvi_p": 4.40610370710529e-59,
  "epsilon_sq_nvi": 0.2966,
  "epsilon_sq_bap": 0.0273,
  "pairwise_bap_holm_p": {
    "Tier3_vs_Tier1": 0.12791113916000985,
    "Tier3_vs_Tier2": 3.558114881020249e-06,
    "Tier2_vs_Tier1": 0.012380203566109183
  },
  "rank_biserial_t3_vs_t2": -0.217,
  "group_medians_ci": {
    "Tier 1 (Basic)": {
      "n": 205,
      "c_day_median": 11.27,
      "c_day_ci95": [
        8.9,
        13.17
      ],
      "bap_media

## H2. Импортная премия vs локальное R&D-производство (контроль по Bio-Tier и категории)

- **Нулевая гипотеза ($H_0$):** H0: При фиксированном технологическом уровне (Bio-Tier) и категории вещества страна производства (Импорт vs РФ/ЕАЭС) не влияет на цену и индекс ценности NVI.
- **Статистический критерий:** `Двухфакторный OLS ANOVA (Type II SS) + Mann-Whitney U`
- **Вердикт:** **H0 отвергается (p < 0.0001)**
- **R&D и Бизнес-вывод:** Даже при строгом статистическом контроле категории вещества и уровня биодоступности (Tier 1–3) через Type-II ANOVA, импортные препараты несут чистую ценовую премию +122.2% (F=228.6, p=3.61e-46; медиана 1050.0 ₽ против 486.0 ₽ у РФ). При этом за счет высокой стоимости курса средний индекс потребительской ценности NVI у российских разработок значимо выше (62.4 против 45.2, p=2.65e-29).

In [3]:
# Результаты статистического теста H2
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H2']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H2",
  "title": "Импортная премия vs локальное R&D-производство (контроль по Bio-Tier и категории)",
  "null_hypothesis": "H0: При фиксированном технологическом уровне (Bio-Tier) и категории вещества страна производства (Импорт vs РФ/ЕАЭС) не влияет на цену и индекс ценности NVI.",
  "test_used": "Двухфакторный OLS ANOVA (Type II SS) + Mann-Whitney U",
  "mannwhitney_price_U": 120786.5,
  "mannwhitney_price_p": 9.324459377721086e-30,
  "mannwhitney_cday_p": 2.8348421693987816e-21,
  "mannwhitney_nvi_p": 2.6509700336951626e-29,
  "anova_import_F": 228.63,
  "anova_import_p": 3.607114813500893e-46,
  "controlled_import_premium_pct": 122.2,
  "rank_biserial_price": 0.4894,
  "group_summary": {
    "Импортные БАДы": {
      "n": 245,
      "median_price_rub": 1050.0,
      "price_ci95": [
        935.0,
        1179.0
      ],
      "median_c_day_rub": 24.83,
      "mean_nvi": 45.2,
      "tier3_share_pct": 18.8
    },
    "Российские БАДы (РФ/ЕАЭС)": {
      "n": 662,
      "me

## H3. Экономика галеновых форм доставки: Gummies и Липосомы против Таблеток и Капсул

- **Нулевая гипотеза ($H_0$):** H0: Галеновая форма выпуска (пастилки Gummies, липосомы/мицеллы, капсулы, таблетки) не влияет на стоимость суточного приема (C_day) и потребительский рейтинг.
- **Статистический критерий:** `Kruskal-Wallis H-test + Mann-Whitney U + Bootstrap 95% ДИ медианы`
- **Вердикт:** **H0 отвергается (p < 0.0001)**
- **R&D и Бизнес-вывод:** Галеновая матрица радикально меняет Unit-экономику (H=26.9, p=6.25e-06): жевательные пастилки Gummies обходятся потребителю в 11.16 ₽/день, а липосомальные/мицеллированные формы — в 33.2 ₽/день против 12.37 ₽/день у классических твердых таблеток (p=8.28e-01). При этом пастилки Gummies демонстрируют высокую приверженность потребителей, но в 62.1% случаев содержат сахар/патоку на первых местах состава — это открывает прямую R&D-нишу для бесcахарных микроинкапсулированных Gummies.

In [4]:
# Результаты статистического теста H3
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H3']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H3",
  "title": "Экономика галеновых форм доставки: Gummies и Липосомы против Таблеток и Капсул",
  "null_hypothesis": "H0: Галеновая форма выпуска (пастилки Gummies, липосомы/мицеллы, капсулы, таблетки) не влияет на стоимость суточного приема (C_day) и потребительский рейтинг.",
  "test_used": "Kruskal-Wallis H-test + Mann-Whitney U + Bootstrap 95% ДИ медианы",
  "kruskal_form_cday_H": 26.88,
  "kruskal_form_cday_p": 6.25111706428091e-06,
  "mannwhitney_gummies_vs_tablets_cday_p": 0.8280268591263782,
  "mannwhitney_gummies_vs_tablets_rating_p": 0.02437682555859099,
  "form_metrics": {
    "Жевательные пастилки (Gummies)": {
      "n": 66,
      "c_day_median": 11.16,
      "c_day_ci95": [
        9.37,
        14.09
      ],
      "mean_bayesian_rating": 4.39,
      "sugar_share_pct": 62.1
    },
    "Липосомы / Мицеллы (R&D)": {
      "n": 22,
      "c_day_median": 33.2,
      "c_day_ci95": [
        25.0,
        54.08
      ],
      "mean_bayesian_rating": 4.24,
      "s

## H4. Технологический аудит эксципиентов: Диоксид титана (E171) в оболочках и Сахар в Gummies

- **Нулевая гипотеза ($H_0$):** H0: Частота присутствия диоксида титана (E171) и сахарно-паточных носителей независима от галеновой формы выпуска и страны происхождения.
- **Статистический критерий:** `Критерий Хи-квадрат Пирсона (χ²) + V Крамера + точный тест Фишера`
- **Вердикт:** **H0 отвергается (p < 0.0001 по обеим матрицам эксципиентов)**
- **R&D и Бизнес-вывод:** Выявлена строгая технологическая поляризация эксципиентов: 1) Диоксид титана E171 (запрещенный EFSA в ЕС с 2022 г.) сконцентрирован в пленочных покрытиях твердых таблеток (31.1% содержат E171) и непрозрачных твердых капсулах (14.3%), тогда как в порошках, пастилках и шипучих формах отсутствует (χ²=81.8, p=5.80e-15, V Крамера=0.300). 2) Сахар и глюкозно-паточные сиропы доминируют в жевательных пастилках Gummies (62.1% SKU) и шипучих таблетках (32.0%) против 2.3% в капсулах (χ²=255.8, p=1.60e-51, V Крамера=0.531).

In [5]:
# Результаты статистического теста H4
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H4']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H4",
  "title": "Технологический аудит эксципиентов: Диоксид титана (E171) в оболочках и Сахар в Gummies",
  "null_hypothesis": "H0: Частота присутствия диоксида титана (E171) и сахарно-паточных носителей независима от галеновой формы выпуска и страны происхождения.",
  "test_used": "Критерий Хи-квадрат Пирсона (χ²) + V Крамера + точный тест Фишера",
  "chi2_form_e171": 81.84,
  "p_form_e171": 5.795749834514104e-15,
  "cramers_v_form_e171": 0.3004,
  "chi2_form_sugar": 255.82,
  "p_form_sugar": 1.5963497951514703e-51,
  "cramers_v_form_sugar": 0.5311,
  "p_fisher_origin_e171": 0.14076344982348432,
  "odds_ratio_origin_e171": 0.712,
  "form_excipient_profile": {
    "Жевательные пастилки (Gummies)": {
      "sku_count": 66,
      "e171_pct": 0.0,
      "sugar_pct": 62.1,
      "mean_k_clean": 0.953
    },
    "Жидкие формы / Капли / Спрей": {
      "sku_count": 77,
      "e171_pct": 5.2,
      "sugar_pct": 14.3,
      "mean_k_clean": 1.003
    },
    "Капсулы (Softgels / Vcap

## H5. Ценовой арбитраж между аптечными агрегаторами (Uteka, Apteka.ru, Zdravcity, Eapteka)

- **Нулевая гипотеза ($H_0$):** H0: Медианная цена идентичных препаратов (парные SKU) одинакова между агрегаторами Apteka.ru, Zdravcity.ru, Eapteka.ru и минимальной ценой Uteka.ru.
- **Статистический критерий:** `Парный непараметрический критерий Фридмана + парный тест Вилкоксона`
- **Вердикт:** **H0 отвергается (p < 0.0001)**
- **R&D и Бизнес-вывод:** На 138 полностью пересекающихся SKU критерий Фридмана фиксирует высокозначимое расхождение цен между агрегаторами (χ²_F=24.8, p=4.04e-06). Среди большой тройки федеральных дистрибьюторов медианная цена на Apteka.ru (935 ₽) статистически значимо ниже, чем на Zdravcity.ru (1064 ₽, p=2.52e-13) и Eapteka.ru (961 ₽, p=2.67e-05), а покупка через лучшую сеть мета-агрегатора Uteka (751 ₽) экономит в среднем 23.8% относительно среднерыночной аптечной полки.

In [6]:
# Результаты статистического теста H5
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H5']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H5",
  "title": "Ценовой арбитраж между аптечными агрегаторами (Uteka, Apteka.ru, Zdravcity, Eapteka)",
  "null_hypothesis": "H0: Медианная цена идентичных препаратов (парные SKU) одинакова между агрегаторами Apteka.ru, Zdravcity.ru, Eapteka.ru и минимальной ценой Uteka.ru.",
  "test_used": "Парный непараметрический критерий Фридмана + парный тест Вилкоксона",
  "matched_all3_skus": 138,
  "friedman_stat": 24.84,
  "friedman_p": 4.042500319895799e-06,
  "wilcoxon_aptekaru_vs_zdravcity_p": 2.522666060765154e-13,
  "wilcoxon_aptekaru_vs_eapteka_p": 2.6670829103939174e-05,
  "wilcoxon_uteka_min_vs_market_avg_p": 1.472307912811607e-110,
  "matched_medians_rub": {
    "Uteka Min Price": 751.0,
    "Apteka.ru": 935.0,
    "Zdravcity.ru": 1063.5,
    "Eapteka.ru": 961.0,
    "Market Avg Price": 1189.5
  },
  "decision": "H0 отвергается (p < 0.0001)",
  "business_insight": "На 138 полностью пересекающихся SKU критерий Фридмана фиксирует высокозначимое расхождение цен между агрегатор